In [ ]:
import pandas as pd
import requests, zipfile, io

In [ ]:
url = 'http://fsck.ch/archive.zip'  # alternative download url
r = requests.get(url)

with zipfile.ZipFile(io.BytesIO(r.content)) as z:
    with z.open('mushroom_cleaned.csv') as f:
        df_download = pd.read_csv(f)

In [ ]:
import s3fs
s3 = s3fs.S3FileSystem()
if not s3.isdir("traindata"):
    s3.mkdir("traindata")

In [ ]:
df_download.to_parquet('s3://traindata/train_raw.parquet', storage_options={"anon": False});

In [ ]:
df = pd.read_parquet('s3://traindata/train_raw.parquet', storage_options={"anon": False});

In [ ]:
df.head(10)

In [ ]:
df.shape

In [ ]:
df.dtypes

In [ ]:
df.nunique()

In [ ]:
df.isna().any()

In [ ]:
df['class'].value_counts()

In [ ]:
categoricals = ['cap-shape', 'gill-attachment', 'gill-color', 'stem-color']
numericals = [c for c in df.drop('class', axis='columns').columns if c not in categoricals]

In [ ]:
for col in df.columns:
    df[col] = df[col].astype('float')

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df.drop('class', axis='columns'), df['class'], random_state=42)

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

clf = GradientBoostingClassifier(n_estimators=10)

In [ ]:
from sklearn.model_selection import cross_val_score

n_folds = 10
scores = cross_val_score(clf, X_train, y_train, cv=n_folds)

print(f"Our classifier has an accuracy of {scores.mean():0.2f}", end=" ")
print(f"(standard deviation {scores.std():0.2f}) over all {n_folds} folds")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

ConfusionMatrixDisplay.from_estimator(clf, X_test, y_test, cmap=plt.cm.Blues)

print(f"Our classifier has an accuracy of {accuracy_score(y_test, y_pred):0.2f}")
plt.show()

In [ ]:
clf.fit(df.drop('class', axis='columns'), df['class'])